# POC : Détection zones inondables — Pikine & Keur Massar
## Fusion AHP classique + Machine Learning + LSTM

Ce notebook reproduit le pipeline complet du POC technique.
Chaque étape peut être exécutée indépendamment.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
print('Environnement OK')
print('Packages disponibles : numpy, pandas, matplotlib, sklearn, xgboost, lightgbm, folium, tensorflow')

## Étape 1 : Génération données synthétiques

In [ ]:
from src.data_generator import create_synthetic_dataset

df = create_synthetic_dataset(n_pixels=5000, random_state=42)
print(f'Dimensions : {df.shape}')
print(f'Taux inondation : {df["inonde"].mean():.1%}')
df.head()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cols_raw = ['altitude_m', 'pente_pct', 'pluie_mm', 'drainage_km_km2', 'sol_type', 'usage_terre']
for ax, col in zip(axes.ravel(), cols_raw):
    for val, label, color in [(0, 'Non-inonde', 'steelblue'), (1, 'Inonde', 'coral')]:
        df[df['inonde'] == val][col].hist(ax=ax, alpha=0.6, bins=25, label=label, color=color)
    ax.set_title(col)
    ax.legend()
plt.suptitle('Distribution des facteurs selon le statut inondation', fontsize=14)
plt.tight_layout()
plt.show()

## Étape 2 : AHP Classique (méthode Saaty)

In [ ]:
from src.ahp import AHPWeights

ahp = AHPWeights(config.CRITERIA)
ahp.set_saaty_matrix(config.SAATY_MATRIX_VALUES)
ahp.calculate_weights()
ahp.print_summary()
poids_ahp = ahp.weights_dict()

In [ ]:
from src.ahp import apply_ahp_weights, classify_susceptibility

# IMPORTANT : on passe les directions pour que altitude/pente/drainage basses = risque haut
df['susceptibilite_ahp'] = apply_ahp_weights(
    df, poids_ahp, config.NORM_MAPPING, directions=config.FACTOR_DIRECTIONS
)
df['ahp_class'] = classify_susceptibility(df['susceptibilite_ahp'].values)
df['ahp_label'] = df['ahp_class'].map(config.CLASS_LABELS)

print('Distribution des classes AHP :')
print(df['ahp_label'].value_counts().sort_index())

## Étape 3 : Entraînement ML (RF + XGBoost + LightGBM + Ensemble)

In [ ]:
from src.ml_models import train_all_models

ml_results = train_all_models(
    df,
    feature_cols=config.FEATURE_COLS,
    models_dir=config.MODELS_DIR,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
)
ml_results['summary']

In [ ]:
ensemble = ml_results['ensemble']
scaler   = ml_results['scaler']
X_scaled = ml_results['X_scaled']
df['proba_ml'] = ensemble.predict_proba(X_scaled)[:, 1]

# Feature importances comparees
fi_rf  = pd.Series(ml_results['rf'].feature_importances_,  index=config.FEATURE_COLS)
fi_xgb = pd.Series(ml_results['xgb'].feature_importances_, index=config.FEATURE_COLS)
fi_lgb = pd.Series(ml_results['lgb'].feature_importances_, index=config.FEATURE_COLS)

pd.DataFrame({'RF': fi_rf, 'XGB': fi_xgb, 'LGB': fi_lgb}).plot(kind='bar', figsize=(10, 4))
plt.title('Feature Importances par modele')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Étape 4 : Comparaison AHP vs ML + Poids hybrides

In [ ]:
from src.hybrid import (ml_weights_to_ahp_names, compute_hybrid_weights,
                        compare_weights, print_weight_analysis)

poids_ml_cols = ml_results['poids_ml_cols']
poids_ml      = ml_weights_to_ahp_names(poids_ml_cols)
poids_hybrid  = compute_hybrid_weights(poids_ahp, poids_ml, config.ALPHA_EXPERT)

df_compare = compare_weights(poids_ahp, poids_ml)
print_weight_analysis(df_compare)
df_compare

In [ ]:
criteria = list(poids_ahp.keys())
x = range(len(criteria))
w = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([i - w for i in x], [poids_ahp[c]*100    for c in criteria], w,
       label='AHP Expert',     color='steelblue',      alpha=0.85)
ax.bar([i     for i in x], [poids_ml[c]*100     for c in criteria], w,
       label='ML Data',        color='coral',           alpha=0.85)
ax.bar([i + w for i in x], [poids_hybrid[c]*100 for c in criteria], w,
       label='Hybrid (final)', color='mediumseagreen',  alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(criteria, rotation=30, ha='right')
ax.set_ylabel('Poids (%)')
ax.set_title('AHP Expert vs ML Data-Driven vs Hybride (70/30)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Étape 5 : Cartes hybrides & validation

In [ ]:
from src.hybrid import build_hybrid_maps, validate_map

df, _ = build_hybrid_maps(
    df, poids_ahp, poids_ml_cols, config.NORM_MAPPING,
    alpha_expert=config.ALPHA_EXPERT,
    class_labels=config.CLASS_LABELS,
)

y_true = df['inonde'].values
metrics_ahp    = validate_map(y_true, df['susceptibilite_ahp'].values,    'AHP Classique')
metrics_hybrid = validate_map(y_true, df['susceptibilite_hybrid'].values, 'Hybrid AHP+ML')
metrics_ml     = validate_map(y_true, df['proba_ml'].values,              'ML Pur')

pd.DataFrame([
    {k: v for k, v in m.items() if k != 'cm'}
    for m in (metrics_ahp, metrics_hybrid, metrics_ml)
])

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 6))
for col, label, color, m in [
    ('susceptibilite_ahp',    'AHP Classique', 'steelblue',      metrics_ahp),
    ('susceptibilite_hybrid', 'Hybrid AHP+ML', 'coral',          metrics_hybrid),
    ('proba_ml',              'ML Pur',         'mediumseagreen', metrics_ml),
]:
    scores = df[col].values
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    fpr, tpr, _ = roc_curve(y_true, scores_norm)
    ax.plot(fpr, tpr, label=f'{label} (AUC={m["auc"]:.3f})', color=color, lw=2)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatoire')
ax.set_xlabel('Taux Faux Positifs')
ax.set_ylabel('Taux Vrais Positifs')
ax.set_title('Courbes ROC — Comparaison des modeles')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Étape 6 : LSTM temporel (prédiction 1 mois à l'avance)

In [ ]:
from src.data_generator import create_timeseries_dataset
from src.lstm_model import train_lstm

df_ts = create_timeseries_dataset(n_zones=50, n_months=60)
print(f'Dataset temporel : {df_ts.shape}  ({df_ts["zone_id"].nunique()} zones x {df_ts["date"].nunique()} mois)')

lstm_results = train_lstm(
    df_ts,
    lookback=config.LOOKBACK,
    epochs=20,
    batch_size=config.LSTM_BATCH_SIZE,
    models_dir=config.MODELS_DIR,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
h = lstm_results['history'].history

# Loss
axes[0].plot(h['loss'],     label='Train Loss')
axes[0].plot(h['val_loss'], label='Val Loss')
axes[0].set_title('LSTM — Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

# AUC — recherche insensible a la casse et sans 'val_'
auc_key = next((k for k in h if 'auc' in k.lower() and not k.lower().startswith('val')), None)
val_key = next((k for k in h if 'auc' in k.lower() and k.lower().startswith('val')), None)
if auc_key and val_key:
    axes[1].plot(h[auc_key], label='Train AUC')
    axes[1].plot(h[val_key], label='Val AUC')
    axes[1].set_title('LSTM — AUC')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f'LSTM final : AUC={lstm_results["auc"]:.3f}  F1={lstm_results["f1"]:.3f}')

## Étape 7 : Carte interactive Folium + rapport complet

In [ ]:
from src.visualization import generate_all_visuals

generate_all_visuals(
    df=df,
    poids_ahp=poids_ahp,
    poids_ml=poids_ml,
    poids_final=poids_hybrid,
    df_compare=df_compare,
    metrics_ahp=metrics_ahp,
    metrics_hybrid=metrics_hybrid,
    metrics_ml=metrics_ml,
    lstm_results=lstm_results,
    ml_summary=ml_results['summary'],
    ahp_cr=ahp.consistency_ratio,
    output_dir=config.OUTPUT_DIR,
)

In [ ]:
# Afficher la carte interactive directement dans le notebook
from IPython.display import IFrame
IFrame(src='../output/carte_pikine_hybrid.html', width='100%', height=500)

## Étape 8 : Fonction d'inférence (pixel unique)

In [ ]:
from src.inference import predict_susceptibility

# Zone urbaine basse — Medina Gounass / Pikine inondable (haut risque)
# sol  : 0=Dunes, 1=Ferrugineux, 2=Halomorphes, 3=Hydromorphes, 4=Eau
# usage: 0=Savane, 1=Culture maraichere, 2=Lac, 3=Mare, 4=Savane arboree, 5=Localite
result = predict_susceptibility(
    altitude=2.5,    # Tres bas (bas-fonds)
    pente=0.2,       # Tres plat
    pluie=88.0,      # Forte pluviometrie (Keur Massar est)
    drainage=0.3,    # Drainage tres faible
    sol=3,           # Hydromorphes (risque max)
    usage=5,         # Localite urbaine dense
    poids_ahp=poids_ahp,
    poids_hybrid=poids_hybrid,
    ensemble_model=ensemble,
    scaler=scaler,
)

print('=== PREDICTION ZONE A HAUT RISQUE ===')
for k, v in result.items():
    print(f'  {k:22s}: {v}')

if result["alerte"]:
    print('
*** ALERTE ROUGE : Zone a haut risque d\'inondation ***')

In [ ]:
from src.inference import batch_predict

# Valeurs reelles : pluie (67-95 mm), altitude (0-21.5 m)
zones = [
    {"altitude": 1.5,  "pente": 0.15, "pluie": 88.0, "drainage": 0.2,  "sol": 3, "usage": 5},
    {"altitude": 12.0, "pente": 1.2,  "pluie": 70.0, "drainage": 0.8,  "sol": 0, "usage": 1},
    {"altitude": 4.0,  "pente": 0.4,  "pluie": 80.0, "drainage": 1.5,  "sol": 1, "usage": 4},
    {"altitude": 0.5,  "pente": 0.05, "pluie": 94.0, "drainage": 0.05, "sol": 4, "usage": 3},
]
noms = ["Pikine bas-fonds", "Dune nord (sane)", "Keur Massar est", "Zone mare"]
predictions = batch_predict(zones, poids_ahp, poids_hybrid, ensemble, scaler)

for nom, pred in zip(noms, predictions):
    print(f"{nom:22s} | AHP={pred['score_ahp']:.0f}/255 | Hybrid={pred['score_hybrid']:.0f}/255 | Classe={pred['classe_hybrid']}")

## Résumé des résultats

| Étape | Description | Statut |
|-------|-------------|--------|
| 1 | Dataset synthétique Pikine (5000 pixels, 6 facteurs) | ✅ |
| 2 | AHP classique Saaty — Consistency Ratio < 0.10 | ✅ |
| 3 | ML : RF + XGBoost + LightGBM + Ensemble Voting | ✅ |
| 4 | Comparaison poids AHP vs ML — divergences analysées | ✅ |
| 5 | Cartes hybrides + validation AUC-ROC | ✅ |
| 6 | LSTM — prédiction temporelle (lookback 6 mois) | ✅ |
| 7 | Graphiques PNG + carte Folium + rapport .txt | ✅ |
| 8 | Inférence opérationnelle pixel unique ou batch | ✅ |